# 08 · Declarative pipelines
*Describe what each table should be and let the platform work out the how.*

In chapters 03 to 06 you wrote the plumbing yourself: checkpoint paths, write modes, the order of the steps, the quality split. Now the same bronze → silver → gold flow is declared in a short Python file
([`pipelines/shield_insight.py`](../../pipelines/shield_insight.py), open it first), and the platform builds the graph, tracks what it already read, and counts every row the quality rules reject.

| Concept | Where you meet it |
|---|---|
| A declarative pipeline: you declare datasets, the engine plans and runs them | the next section, then Steps 1 and 2 |
| Streaming table (incremental) and materialized view (batch result kept fresh) | the next section, then Steps 2 to 4 |
| Expectations: quality rules attached to a dataset (warn, drop or fail) | the next section, then Steps 3 and 5 |
| The pipeline event log | Step 5 |
| An update only processes what is new | Step 6 |

Deep dive: [docs/08-declarative-pipelines.md](../../docs/08-declarative-pipelines.md)

**Needs:**
* Chapter 03 delivered the JSON files into `/Volumes/shield/landing/raw/incident_reports` (all five). The pipeline reads those files itself with its own checkpoint, so it does **not** need the bronze, silver or gold tables from chapters 03 to 06.
* A workspace on the **Premium** plan with Unity Catalog, serverless compute for pipelines, and permission to create a schema in `shield` and a pipeline. Pipelines are a GA product (formerly Delta Live Tables); nothing used here is in preview.
* Open this notebook from a Git folder, because it imports the quality rules from [`src/shield_data/quality.py`](../../src/shield_data/quality.py) (see chapter 00).

Steps 1, 2 and 6 are clicks in the workspace UI, not code; the code cells check what the UI steps produced.
The **Expected** numbers assume the dataset from chapter 02 (10,000 incidents, seed 42, 5 % dirty, 250 replayed duplicates, all five files delivered); other data gives other numbers and a ❌, which is not a bug.
Nothing here has been run in a workspace yet, so every **Expected** is an expectation, not a recorded result, and the UI labels come from Microsoft Learn and can change.

## Declarative pipelines, streaming tables, materialized views and expectations

**The idea:** in a notebook you write *how*: read this, write that, then run the next cell. In a **declarative pipeline** you only declare *what* each dataset is, as a function that returns a DataFrame.
The engine reads all the declarations, works out the dependency graph from which dataset reads which, and runs them in the right order, with checkpoints and retries. It is a framework (Lakeflow Spark Declarative Pipelines, formerly Delta Live Tables), not a pattern.
Pipeline code is not a notebook: the decorators only work inside a pipeline, which is why the code lives in a `.py` file and this notebook is the companion that creates, runs and inspects it.

```
 /Volumes/shield/landing/raw/incident_reports      JSON files delivered in chapter 03
        │  Auto Loader (chapter 03)
        ▼
 incidents_bronze           streaming table      @dp.table              as received + _source_file, _ingested_at
        │  parse reported_ts, dropDuplicates(incident_id)
        │  expectations (expect_or_drop): has_agent, valid_timestamp, valid_severity, non_negative_casualties
        ▼
 incidents_silver           streaming table      @dp.table              typed, valid, one row per incident
        │  group by date, type, severity
        ▼
 daily_threat_briefing      materialized view    @dp.materialized_view  incidents and casualties per day
                                                                        (all three in shield.pipeline)
```

| Piece | What it is | Used for |
|---|---|---|
| **Streaming table**, `@dp.table` on a *streaming* read | a Delta table that is fed incrementally: each update processes only data it has not seen | ingestion and row-by-row transforms (bronze, silver) |
| **Materialized view**, `@dp.materialized_view` on a *batch* read | a table defined by a query; the engine keeps it equal to the query result, incrementally when it can | joins and aggregates (gold) |
| **Expectation**, `@dp.expect` / `expect_or_drop` / `expect_or_fail` | a named SQL condition on a dataset: keep the row but count it (warn), drop it, or fail the update | quality rules (the idea from chapter 06) |
| **Event log** | a table of everything that happened in each update, including expectation counts | monitoring |

**How this notebook implements it**

* **The code** is `pipelines/shield_insight.py`: `incidents_bronze` reads the raw files with Auto Loader; `incidents_silver` reads bronze as a stream, parses `reported_ts`, drops duplicates by `incident_id` and applies the four rules from `INCIDENT_RULES` as `expect_or_drop` expectations; `daily_threat_briefing` aggregates silver.
* **The pipeline** is created in the UI (Step 1) with default catalog `shield` and default schema `pipeline`, so the three datasets land in `shield.pipeline` and nothing in `shield.bronze`, `shield.silver` or `shield.gold` is touched.
* **The notebook** only reads: it checks the tables (Steps 2 to 4), the event log (Step 5) and a second update (Step 6).

**How it differs from chapters 03 to 06** (so the numbers don't surprise you)

* Bad rows are dropped and only *counted* in the event log; there is no quarantine table (that is a mini-project in the doc).
* Duplicates are removed by `dropDuplicates`, which keeps the first copy it sees (chapters 05 and 06 keep the latest). The copies are identical here, so the result is the same. The state this needs grows without bound; production code adds a watermark.
* The gold view has no `city` column: the pipeline does not read the locations feed, so it is not the same table as `shield.gold.daily_threat_briefing`.

**Why it helps:** less code, no checkpoint paths or write modes to get wrong, automatic ordering, retries and incremental processing, and quality numbers for free. The price is less control and one more thing to learn.

**Not covered yet:** scheduling the pipeline as a job task (chapter 10), deploying it as code (chapter 13), change data capture (`AUTO CDC`), and sinks. The AI-assisted pipeline editor (Genie Code) is in Public Preview and not used.

## Setup

In [ ]:
def show_checks(checks):
    """Show (check, result, expected) rows with a ✅ / ❌ for whether result matches expected."""
    fmt = lambda v: f"{v:,}" if isinstance(v, int) and not isinstance(v, bool) else str(v)
    rows = [(what, fmt(got), fmt(want), "✅" if got == want else "❌") for what, got, want in checks]
    display(spark.createDataFrame(rows, "check string, result string, expected string, ok string"))


import os
import sys

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..", "..", "src")))
from pyspark.sql import functions as F

from shield_data.quality import INCIDENT_RULES

PIPELINE_SCHEMA = "shield.pipeline"
RAW_INCIDENTS = "/Volumes/shield/landing/raw/incident_reports"

assert dbutils.fs.ls(RAW_INCIDENTS), "Run notebook 03 first: no incident files in the raw landing zone."
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {PIPELINE_SCHEMA}")

## 1 · Create the pipeline

### Step 1 · Create the pipeline (UI)
**Goal:** a pipeline named `shield_insight` whose only source file is `pipelines/shield_insight.py`.

**Expected:** after **Dry run** (it validates the code without updating any data) the pipeline graph shows three nodes: `incidents_bronze → incidents_silver → daily_threat_briefing`, and the Issues panel shows no errors.

In the workspace (labels from Microsoft Learn, not checked in this workspace):

1. In the sidebar click **New → ETL pipeline** (or **Jobs & Pipelines → New → ETL pipeline**). The defaults are Unity Catalog and serverless compute.
2. Name it `shield_insight`. Click the catalog and schema shown next to the name and choose catalog `shield`, schema `pipeline` (the setup cell created it).
3. Add the code: in the menu next to **Use sample code** choose **Add existing source code** and pick `shield_insight.py` in the `pipelines` folder of your Git folder. (Alternative: **Settings → Source code → Configure paths → Add path**.)
   The new pipeline starts with a blank `my_transformation` file. It is harmless; if you chose the sample code instead, remove those files from the pipeline, because they define example tables.
4. Optional: **Settings → Configuration**, add `raw_path = /Volumes/shield/landing/raw`. The code uses that same path when the key is missing.
5. Click **Dry run**.

There is no code cell for this step.

### Step 2 · Run it once
**Goal:** the first update, which ingests everything in the landing zone.

**Expected:** click **Run pipeline**. The pipeline is triggered, not continuous, so it processes what is there and stops. The update finishes in a few minutes (not measured). Then `SHOW TABLES IN shield.pipeline` lists the three datasets (the event log is hidden) and `incidents_bronze` holds 10,250 rows: 10,000 incidents plus 250 replayed deliveries, which bronze keeps as received.

What you will see:

| database | tableName | isTemporary |
|---|---|---|
| pipeline | daily_threat_briefing | false |
| pipeline | pipeline | incidents_bronze | false |
| pipeline | incidents_silver | false |

The cell only reads, so it is safe to re-run after any update.

In [ ]:
raw_files = [file.name for file in dbutils.fs.ls(RAW_INCIDENTS)]
pipeline_tables = spark.sql(f"SHOW TABLES IN {PIPELINE_SCHEMA}")
display(pipeline_tables)

datasets = ["daily_threat_briefing", "incidents_bronze", "incidents_silver"]
present = sorted(row["tableName"] for row in pipeline_tables.collect() if row["tableName"] in datasets)
assert present, "Create and run the pipeline first (Steps 1 and 2)."

bronze_rows_first_update = spark.table(f"{PIPELINE_SCHEMA}.incidents_bronze").count()
show_checks([
    ("files in the raw landing zone", len(raw_files), 5),
    ("pipeline datasets present", present, datasets),
    ("incidents_bronze rows", bronze_rows_first_update, 10_250),
])

## 2 · What the pipeline built

### Step 3 · Silver: only valid, unique incidents
**Goal:** prove the `expect_or_drop` expectations did the work that chapter 06 did by hand.

| Input | Value |
|---|---|
| `shield.pipeline.incidents_silver` | built by the pipeline from bronze |
| `INCIDENT_RULES` | the four rules from `src/shield_data/quality.py`, which the pipeline file repeats as expectations |

**Expected:** 9,500 rows, one per `incident_id`, and not one row that fails any of the four rules. The severity breakdown has the four allowed values only (no `catastrophic`):

| severity | n |
|---|---|
| low | 4,722 |
| medium | 2,915 |
| high | 1,428 |
| critical | 435 |

Chapter 06's silver table has the same 9,500 incidents, so if you ran it the two routes agree. The cell only reads.

In [ ]:
silver_incidents = spark.table(f"{PIPELINE_SCHEMA}.incidents_silver")
display(silver_incidents.groupBy("severity").count().orderBy(F.desc("count")))

all_rules_pass = " AND ".join(f"coalesce({condition}, false)" for condition in INCIDENT_RULES.values())
show_checks([
    ("silver rows", silver_incidents.count(), 9_500),
    ("distinct incident_id", silver_incidents.select("incident_id").distinct().count(), 9_500),
    ("rows failing any rule", silver_incidents.filter(f"NOT ({all_rules_pass})").count(), 0),
    ("distinct severities", sorted(row["severity"] for row in silver_incidents.select("severity").distinct().collect()), ["critical", "high", "low", "medium"]),
])

### Step 4 · Gold: a materialized view
**Goal:** check the aggregate the engine keeps up to date.

**Expected:** one row per `(report_date, incident_type, severity)`: 4,975 rows. The incidents add up to the silver rows (9,500) and the casualties to 5,681. The display shows the ten rows with the latest dates, so only a sample is cut.
A materialized view is refreshed by each update, incrementally when the query allows it and by recomputing otherwise; you get the same result either way.

What you will see (illustrative):

| report_date | incident_type | severity | incidents | casualties |
|---|---|---|---|---|
| 2025-12-31 | breach | low | … | … |
| … | … | … | … | … |

The cell only reads.

In [ ]:
daily_briefing = spark.table(f"{PIPELINE_SCHEMA}.daily_threat_briefing")
display(daily_briefing.orderBy(F.desc("report_date")).limit(10))

totals = daily_briefing.agg(F.sum("incidents").alias("incidents"), F.sum("casualties").alias("casualties")).first()
show_checks([
    ("rows (date, type, severity)", daily_briefing.count(), 4_975),
    ("incidents in the briefing", totals["incidents"], silver_incidents.count()),
    ("casualties in the briefing", totals["casualties"], 5_681),
])

## 3 · Quality numbers and incremental runs

### Step 5 · Expectation metrics in the event log
**Goal:** see how many rows each rule rejected. The pipeline dropped them, so these numbers are the only trace.

| Input | Value |
|---|---|
| `event_log(TABLE(shield.pipeline.incidents_silver))` | the pipeline's event log. Only the pipeline owner can query it, with a SQL warehouse or compatible compute; if the query is refused, run it in the SQL editor as the owner |

**Expected:** four expectations named exactly like the keys of `INCIDENT_RULES`. Over all updates they rejected 500 rows, one per defective incident: `has_agent` 95, `valid_timestamp` 205, `valid_severity` 109, `non_negative_casualties` 91.
That assumes the expectations see the rows *after* `dropDuplicates`. If your numbers are close to chapter 06's (96 / 209 / 110 / 96, which include the replayed duplicates), the duplicates were counted too. If every number is about double, the sum counts the final progress event as well as the per-batch ones. Both readings are unverified until the notebook runs in a workspace.
Event delivery is best effort, so a missing event is possible.

What you will see (illustrative):

| expectation | passed | failed |
|---|---|---|
| has_agent | … | 95 |
| valid_timestamp | … | 205 |
| … | … | … |

The cell only reads.

In [ ]:
expectation_metrics = spark.sql(
    f"""
    SELECT expectation.name AS expectation,
           SUM(expectation.passed_records) AS passed,
           SUM(expectation.failed_records) AS failed
    FROM (
      SELECT explode(
               from_json(
                 details:flow_progress.data_quality.expectations,
                 'array<struct<name: string, dataset: string, passed_records: bigint, failed_records: bigint>>'
               )
             ) AS expectation
      FROM event_log(TABLE({PIPELINE_SCHEMA}.incidents_silver))
      WHERE event_type = 'flow_progress' AND details:flow_progress.data_quality.expectations IS NOT NULL
    )
    GROUP BY expectation.name
    ORDER BY expectation.name
    """
)
display(expectation_metrics)

failed_by_rule = {row["expectation"]: row["failed"] for row in expectation_metrics.collect()}
show_checks([
    ("expectations in the event log", sorted(failed_by_rule), sorted(INCIDENT_RULES)),
    ("has_agent failed", failed_by_rule.get("has_agent"), 95),
    ("valid_timestamp failed", failed_by_rule.get("valid_timestamp"), 205),
    ("valid_severity failed", failed_by_rule.get("valid_severity"), 109),
    ("non_negative_casualties failed", failed_by_rule.get("non_negative_casualties"), 91),
])

### Step 6 · Run it again: nothing new to process (UI)
**Goal:** see that an update does only the new work. The streaming tables remember which files they have read.

**Expected:** click **Run pipeline** a second time with no new files in the landing zone. The update finishes quickly, `incidents_bronze` still has 10,250 rows (not 20,500), silver still has 9,500, and the event log has at least two `create_update` events (one per update).
To watch it pick up new files you need files it has not seen: if you ran chapter 03 only partly, finish chapter 03 (it delivers all five files) and run the pipeline again; or try the replay mini-project in the doc.
Do not trigger **Full refresh** here unless you mean to: it clears the streaming tables and reads every file again.

What you will see:

| check | result | expected | ok |
|---|---|---|---|
| incidents_bronze rows | 10,250 | 10,250 | ✅ |

In [ ]:
updates_in_event_log = spark.sql(
    f"SELECT COUNT(*) AS updates FROM event_log(TABLE({PIPELINE_SCHEMA}.incidents_silver)) WHERE event_type = 'create_update'"
).first()["updates"]

show_checks([
    ("updates in the event log (at least 2)", updates_in_event_log >= 2, True),
    ("incidents_bronze rows", spark.table(f"{PIPELINE_SCHEMA}.incidents_bronze").count(), bronze_rows_first_update),
    ("incidents_silver rows", spark.table(f"{PIPELINE_SCHEMA}.incidents_silver").count(), 9_500),
])

## 👀 In the workspace

* **Jobs & Pipelines** → `shield_insight`: the pipeline graph (three nodes), the list of updates, and the duration of each.
* Click `incidents_silver` → **Data quality** tab: the same expectation counts as Step 5.
* **Catalog** → `shield` → `pipeline`: the two streaming tables and the materialized view. Lineage should show `incidents_bronze` feeding `incidents_silver` feeding `daily_threat_briefing`.

## 🧹 Clean up

The pipeline is not continuous, so it costs nothing while idle. Nothing else needs it: chapters 10 and 16 use the tables from chapters 03 to 06, and chapter 13 deploys its own copy of this pipeline as a bundle (named like `shield_insight_dev`, writing to schema `shield.pipeline_dev`).
To remove this one, delete the pipeline in **Jobs & Pipelines** (its tables should go with it, which has not been checked; look in Catalog afterwards), then run the commented cell for anything left.

In [ ]:
# spark.sql("DROP SCHEMA IF EXISTS shield.pipeline CASCADE")

## 🎯 Recap

* A declarative pipeline is a set of dataset declarations; the engine builds the graph, tracks progress and retries. Streaming tables take the incremental steps, materialized views the aggregates.
* Expectations are quality rules attached to a dataset. Here they dropped the 500 defective incidents (95 + 205 + 109 + 91) and counted them in the event log, so silver has 9,500 incidents without a line of splitting code.
* A second update processed nothing new, because the streaming tables remember what they read.
* The trade-off is control: no quarantine table unless you build one, and `dropDuplicates` state that grows.

Mini-projects: [docs/08-declarative-pipelines.md](../../docs/08-declarative-pipelines.md#try-it-yourself)

> 💡 Next: chapter 09 moves from files to true streams with Structured Streaming and Event Hubs.